# 02. Full 파인튜닝 실습: Qwen2.5-0.5B를 "고양이 비서"로 만들기

1단계에서는 순수 PyTorch로 작은 GPT를 직접 만들고 프리트레이닝과 파인튜닝을 해 봤습니다.
이번에는 **이미 프리트레이닝이 끝난 진짜 LLM**(Qwen2.5-0.5B, 약 5억 개 파라미터)을 Hugging Face로 불러와서 **모든 가중치를 학습하는 Full 파인튜닝**을 합니다.

**목표:** 질문을 받으면 친절한 고양이 비서 "냥이"의 말투(`~다냥!`, 끝에 `🐾`)로 짧게 답하는 모델 만들기.
말투는 학습 전후 차이가 눈에 확 보이기 때문에 파인튜닝 효과를 확인하기 좋은 과제입니다.

**이 노트북에서 배우는 것**

1. Hugging Face의 3요소: 토크나이저, 모델, 데이터셋
2. 학습 전 베이스 모델은 어떻게 답하는가
3. SFT 데이터 만들기: 프롬프트 형식, 답변 부분만 loss 계산(`-100` 마스킹), 패딩
4. `Trainer`로 학습하기: 1단계에서 직접 짠 학습 루프와 무엇이 같은가
5. 메모리 절약: 혼합 정밀도(bf16/fp16), gradient accumulation, gradient checkpointing
6. 학습 전후 비교, 그리고 catastrophic forgetting 확인
7. 모델 저장하고 다시 불러오기

**실행 환경**

| 환경 | 예상 시간 | 필요 메모리 |
|---|---|---|
| RTX 3060 12GB | 약 1~3분 (학습) | VRAM 약 8~10GB |
| CPU만 있는 경우 | 약 10~30분 (스텝 수를 줄인 설정) | RAM 12GB 이상 권장 |

처음 실행할 때 Hugging Face Hub에서 모델(약 1GB)을 내려받으므로 인터넷 연결이 필요합니다.

> 💡 셀을 위에서부터 순서대로 실행하세요 (`Shift + Enter`). 중간에 GPU 메모리가 꼬이면 커널을 재시작(`Kernel → Restart`)하고 처음부터 다시 실행하면 됩니다.

## 0. 환경 확인과 실행 설정

GPU가 있으면 GPU 설정, 없으면 CPU 설정을 자동으로 고릅니다.

**혼합 정밀도(mixed precision)란?**
가중치는 fp32(4바이트)로 보관하되, 순전파/역전파의 계산은 16비트로 해서 속도를 올리고 activation 메모리를 줄이는 방법입니다.

- **bf16:** fp32와 표현 범위가 같아서 안정적입니다. RTX 30xx(Ampere) 이상에서 지원하므로 **RTX 3060은 bf16을 씁니다.**
- **fp16:** 더 오래된 GPU(RTX 20xx 등)용. 값이 너무 작아지면 0이 되는 문제(underflow)가 있어서 loss scaling이 함께 쓰입니다. `Trainer`가 자동으로 처리합니다.
- **CPU:** 16비트 연산 이득이 작아서 fp32로 학습합니다.

In [ ]:
import json
import math
import os
import time
from pathlib import Path

import torch
import transformers
import datasets

print("PyTorch      :", torch.__version__)
print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)

if torch.cuda.is_available():
    device = "cuda"                      # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"                       # Apple Silicon Mac (M1~M4) GPU
else:
    device = "cpu"
print("사용할 장치  :", device)

if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU 이름     :", props.name)
    print(f"VRAM         : {props.total_memory / 1024**3:.1f} GB")
    print("bf16 지원    :", torch.cuda.is_bf16_supported())

transformers.set_seed(42)

아래 `config`가 이 노트북의 모든 학습 설정입니다. 값을 바꿔 가며 실험해 보세요.

- **effective batch size** = `batch_size × grad_accum_steps`. GPU 설정은 4 × 2 = 8개 예시를 보고 가중치를 한 번 업데이트합니다.
- **학습률(lr)**: 1단계의 TinyGPT는 `1e-3`이었지만, 프리트레이닝된 큰 모델은 훨씬 작은 `1e-5 ~ 5e-5`를 씁니다. 크게 잡으면 원래 배운 지식이 쉽게 망가집니다.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # 프리트레이닝만 된 베이스 모델 (대화 튜닝 전)

if device in ("cuda", "mps"):
    # Mac(mps)은 GPU 설정을 쓰되, 혼합 정밀도(bf16/fp16) 학습은 MPS에서 불안정할 수 있어 fp32로 학습한다
    use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
    config = dict(
        max_length=256,          # 한 예시의 최대 토큰 수
        batch_size=4,            # GPU에 한 번에 올리는 예시 수
        grad_accum_steps=2,      # 이만큼 모아서 한 번 업데이트
        epochs=5,                # 학습 데이터를 몇 번 반복할지
        lr=3e-5,
        bf16=use_bf16,
        fp16=device == "cuda" and not use_bf16,
        gradient_checkpointing=False,  # VRAM이 부족(OOM)하면 True로
    )
else:
    config = dict(
        max_length=128,
        batch_size=2,
        grad_accum_steps=4,
        epochs=2,                # CPU는 느리므로 반복 횟수를 줄임
        lr=5e-5,                 # 스텝이 적은 대신 학습률을 조금 크게
        bf16=False,
        fp16=False,
        gradient_checkpointing=False,
    )
    torch.set_num_threads(os.cpu_count())

for k, v in config.items():
    print(f"{k:24}: {v}")
print(f"{'effective batch size':24}: {config['batch_size'] * config['grad_accum_steps']}")

> 📝 **CPU에서 RAM이 부족하다면?** Full 파인튜닝은 0.5B 모델이라도 가중치 + gradient + AdamW 상태로 약 8GB, 여기에 activation이 더 필요합니다.
> RAM이 8GB 이하라면 `MODEL_NAME`을 더 작은 `"HuggingFaceTB/SmolLM2-135M"`(1.35억 파라미터, 영어 위주)으로 바꿔서 흐름만 따라가 보세요.
> 한국어 말투는 덜 자연스럽지만 학습 과정은 똑같습니다.

## 1. Hugging Face의 3요소: 토크나이저, 모델, 데이터셋

1단계에서 직접 만든 것들이 Hugging Face에서는 이렇게 대응됩니다.

| 1단계 (직접 구현) | 2단계 (Hugging Face) |
|---|---|
| `encode` / `decode` 문자 토크나이저 | `AutoTokenizer` (BPE, 어휘 약 15만 개) |
| `TinyGPT` (2레이어, 0.1M) | `AutoModelForCausalLM` (24레이어, 494M) |
| `pretrain_text` 문자열 | `datasets.Dataset` |
| `train()` 학습 루프 | `Trainer` |

### 1-1. 토크나이저

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("어휘 크기    :", len(tokenizer))
print("EOS 토큰     :", repr(tokenizer.eos_token), tokenizer.eos_token_id)
print("PAD 토큰     :", repr(tokenizer.pad_token), tokenizer.pad_token_id)

sample = "고양이는 창가에서 잔다."
ids = tokenizer.encode(sample)
print("\n원문     :", sample)
print("토큰 ID  :", ids)
print("토큰들   :", [tokenizer.decode([i]) for i in ids])
print("다시 복원:", tokenizer.decode(ids))

> 📝 토큰에 `�`가 보이면 한글 한 글자(UTF-8로 3바이트)가 **여러 토큰에 걸쳐 나뉜 것**입니다. Qwen은 바이트 단위 BPE를 쓰기 때문에 드문 글자는 바이트 조각으로 쪼개집니다. 이어 붙여서 decode하면 원래 글자로 돌아옵니다.

1단계의 문자 토크나이저는 이 문장을 13개 토큰으로 나눴지만, Qwen 토크나이저는 자주 나오는 글자 묶음을 하나로 합쳐서 훨씬 적은 토큰을 씁니다.

**EOS(End Of Sequence) 토큰**은 "여기서 글이 끝난다"는 신호입니다. 답변 끝에 EOS를 붙여서 학습해야 모델이 **답을 마치고 멈추는 법**을 배웁니다.
PAD 토큰이 없는 모델도 있는데, 그럴 때는 EOS를 PAD로 같이 씁니다.

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("PAD 토큰이 없어서 EOS를 PAD로 사용합니다.")

### 1-2. 모델

`AutoModelForCausalLM`은 "다음 토큰 예측(causal LM)"용 모델을 불러옵니다. 1단계의 `TinyGPT`와 같은 일을 하는 훨씬 큰 모델입니다.

가중치는 **fp32로 불러옵니다.** 학습 중 계산만 bf16으로 하고(혼합 정밀도), 업데이트되는 원본 가중치는 fp32로 유지해야 작은 학습률의 미세한 변화가 사라지지 않습니다.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)   # 기본값 fp32
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 파라미터  : {n_params / 1e6:.1f}M")
print(f"학습할 파라미터: {n_trainable / 1e6:.1f}M  ← Full 파인튜닝은 전부 학습합니다")
print("가중치 dtype   :", next(model.parameters()).dtype)
print(f"레이어 수: {model.config.num_hidden_layers}, 임베딩 차원: {model.config.hidden_size}")

if device == "cuda":
    print(f"\n모델만 올렸을 때 VRAM 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

### 1-3. HF 모델도 결국 "logits + cross entropy"

1단계에서 `model(x, y)`가 logits와 loss를 돌려줬던 것처럼, HF 모델도 `labels`를 넘기면 loss를 계산해 줍니다.
한 가지 차이: **HF 모델은 내부에서 labels를 한 칸 밀어 줍니다.** 그래서 `labels`에 `input_ids`를 그대로 넣으면 됩니다.

직접 한 칸 밀어서 계산한 값과 같은지 확인해 봅시다.

In [ ]:
import torch.nn.functional as F

batch = tokenizer(["오늘은 날씨가 맑고 하늘이 파랗다."], return_tensors="pt").to(device)

with torch.no_grad():
    out = model(**batch, labels=batch["input_ids"])

# 1단계 방식: 입력은 앞에서부터, 정답은 한 칸 뒤에서부터
logits = out.logits[:, :-1, :]
targets = batch["input_ids"][:, 1:]
manual_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), targets.reshape(-1))

print("logits 모양       :", tuple(out.logits.shape), "(배치, 토큰 수, 어휘 크기)")
print(f"HF가 계산한 loss  : {out.loss.item():.4f}")
print(f"직접 계산한 loss  : {manual_loss.item():.4f}")
print(f"무작위 기준 ln(V) : {math.log(len(tokenizer)):.2f}  ← 프리트레이닝된 모델이라 훨씬 낮음")

## 2. 학습 데이터 살펴보기

`data/` 폴더에 고양이 비서 말투의 질문/답변 데이터를 준비해 두었습니다. 학습용 42개, 평가용 6개입니다.
한 줄에 예시 하나씩 들어 있는 **JSONL** 형식이고, 직접 열어서 수정하거나 추가해도 됩니다. (3단계 LoRA에서도 같은 데이터를 씁니다.)

실제 파인튜닝 데이터도 이렇게 생겼습니다. 규모만 수천~수만 개로 커질 뿐입니다.

In [ ]:
from datasets import load_dataset

# 노트북은 notebooks/ 폴더에서 실행되므로 한 단계 위의 data/ 폴더를 찾는다
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")

raw = load_dataset(
    "json",
    data_files={
        "train": str(DATA_DIR / "cat_assistant_train.jsonl"),
        "eval": str(DATA_DIR / "cat_assistant_eval.jsonl"),
    },
)
print(raw)
for ex in raw["train"].select(range(3)):
    print("\n질문:", ex["question"])
    print("답변:", ex["answer"])

## 3. 학습 전 베이스 모델은 어떻게 답할까?

모델에 넣을 **프롬프트 형식**을 정합니다. 학습할 때와 사용할 때 **반드시 같은 형식**을 써야 합니다.

```
### 질문:
{질문}

### 답변:
{답변}<EOS>
```

> 📝 **chat template:** Qwen2.5-Instruct 같은 대화형 모델은 `<|im_start|>user ...` 같은 고유 형식(chat template)이 정해져 있고 `tokenizer.apply_chat_template()`로 만듭니다.
> 여기서는 원리를 보기 위해 **베이스 모델 + 직접 정한 단순한 형식**을 씁니다. 형식이 무엇이든, 학습과 사용 시 일관되기만 하면 됩니다.

In [ ]:
def build_prompt(question):
    return f"### 질문:\n{question}\n\n### 답변:\n"

@torch.no_grad()
def ask(model, question, max_new_tokens=80):
    model.eval()
    inputs = tokenizer(build_prompt(question), return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,                       # 가장 확률 높은 토큰만 고름 (매번 같은 결과)
        repetition_penalty=1.1,                # 같은 말 반복을 약간 억제
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]   # 프롬프트 뒤에 새로 생성된 부분만
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 학습 데이터에 없는 질문들로 테스트
test_questions = [
    "고양이는 몇 시간 자?",        # 평가용 데이터에 있는 질문 (학습에는 안 씀)
    "서울에서 부산까지 어떻게 가?", # 처음 보는 질문
    "LLM이 뭐야?",                  # 처음 보는 질문
    "오늘 저녁 메뉴 추천해 줘.",    # 처음 보는 질문
]

before_answers = {}
for q in test_questions:
    before_answers[q] = ask(model, q)
    print(f"Q: {q}\nA: {before_answers[q]}\n" + "-" * 60)

베이스 모델은 프리트레이닝으로 언어와 지식은 알지만 "질문에 이런 말투로 짧게 답한다"는 것은 모릅니다.
엉뚱한 내용을 이어 쓰거나, 끝내지 못하고 `max_new_tokens`까지 계속 생성하는 경우가 많습니다.

## 4. SFT 데이터 만들기: 토큰화, 답변만 loss, 패딩

1단계 마지막에서 배운 것을 여기서 실제로 씁니다.

1. 프롬프트와 답변을 각각 토큰화해서 이어 붙이고, 끝에 EOS를 붙입니다.
2. **labels**는 `input_ids`를 복사하되 **프롬프트 부분을 `-100`으로** 바꿉니다. → 질문을 외우지 않고 답변하는 법만 배웁니다.
3. 배치 안의 예시들은 길이가 다르므로, 가장 긴 것에 맞춰 PAD를 채웁니다(**패딩**). 채운 부분은 `attention_mask=0`, `labels=-100`으로 둬서 학습에 영향을 주지 않게 합니다.

```
토큰     :  ### 질문 : \n 하늘 은 … ### 답변 : \n  하늘 이  … 🐾  <EOS> <PAD> <PAD>
labels   :  -100 -100 …              -100 -100   하늘 이  … 🐾  <EOS> -100  -100
attn_mask:  1    1    …              1    1      1    1   … 1   1     0     0
```

In [ ]:
def tokenize_example(example):
    prompt_ids = tokenizer(build_prompt(example["question"]), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(example["answer"], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

    input_ids = (prompt_ids + answer_ids)[: config["max_length"]]
    labels = ([-100] * len(prompt_ids) + answer_ids)[: config["max_length"]]
    return {"input_ids": input_ids, "labels": labels}

tokenized = raw.map(tokenize_example, remove_columns=raw["train"].column_names)
print(tokenized)

ex = tokenized["train"][0]
print("\n토큰 수:", len(ex["input_ids"]))
print("loss를 계산하는 토큰 수:", sum(l != -100 for l in ex["labels"]))
print("loss를 계산하는 부분:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

lengths = [len(x) for x in tokenized["train"]["input_ids"]]
print(f"\n학습 데이터 길이: 최소 {min(lengths)}, 평균 {sum(lengths) / len(lengths):.0f}, 최대 {max(lengths)} 토큰")

이제 여러 예시를 하나의 배치로 묶는 **data collator**를 직접 만듭니다. `Trainer`가 배치를 만들 때마다 이 함수를 부릅니다.

In [ ]:
def collate(examples):
    max_len = max(len(ex["input_ids"]) for ex in examples)
    input_ids, labels, attention_mask = [], [], []
    for ex in examples:
        n_pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * n_pad)
        labels.append(ex["labels"] + [-100] * n_pad)
        attention_mask.append([1] * len(ex["input_ids"]) + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }

demo = collate([tokenized["train"][i] for i in range(3)])
for k, v in demo.items():
    print(f"{k:15}: {tuple(v.shape)}")
print("\n첫 예시의 attention_mask 끝부분:", demo["attention_mask"][0, -8:].tolist())
print("첫 예시의 labels 끝부분        :", demo["labels"][0, -8:].tolist())

> 💡 **왜 PAD 위치의 labels를 `-100`으로 두나?** 여기서는 PAD 토큰과 EOS 토큰이 같은 토큰입니다.
> 만약 "labels에서 PAD ID를 전부 -100으로 바꾸기" 방식을 쓰면, 답변 끝의 **진짜 EOS까지 지워져서** 모델이 멈추는 법을 못 배웁니다.
> 그래서 토큰 ID가 아니라 **위치(길이)** 를 기준으로 패딩 부분만 지웁니다. 실제로 자주 나오는 실수입니다.

## 5. `Trainer`로 Full 파인튜닝

`Trainer`는 1단계에서 직접 짠 학습 루프를 대신 돌려 주는 도구입니다. 안에서 하는 일은 같습니다.

```python
for step in range(max_steps):
    batch = 다음 배치                                # DataLoader + collate
    with autocast(bf16):                             # 혼합 정밀도
        loss = model(**batch).loss / grad_accum_steps
    loss.backward()
    if (step + 1) % grad_accum_steps == 0:           # gradient accumulation
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
```

여기에 로깅, 평가, 체크포인트 저장, learning rate 스케줄(warmup 후 감소)까지 해 줍니다.

### Gradient accumulation이란?

배치 크기를 키우면 학습이 안정되지만 activation 메모리가 배치 크기에 비례해서 늘어납니다.
**작은 배치로 여러 번 backward해서 gradient를 모은 다음 한 번만 업데이트**하면, 메모리는 작은 배치만큼 쓰면서 큰 배치와 거의 같은 효과를 냅니다.

In [ ]:
from transformers import Trainer, TrainingArguments

OUTPUT_DIR = Path("../outputs/02_full_ft") if Path("../data").exists() else Path("outputs/02_full_ft")

if config["gradient_checkpointing"]:
    model.config.use_cache = False   # gradient checkpointing과 KV 캐시는 같이 못 씀

steps_per_epoch = math.ceil(len(tokenized["train"]) / (config["batch_size"] * config["grad_accum_steps"]))
total_steps = steps_per_epoch * config["epochs"]
print(f"epoch당 업데이트 횟수: {steps_per_epoch}, 전체: {total_steps}번")

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=config["epochs"],
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    gradient_accumulation_steps=config["grad_accum_steps"],
    learning_rate=config["lr"],
    lr_scheduler_type="cosine",      # 학습률을 서서히 줄임
    warmup_steps=max(1, int(total_steps * 0.1)),  # 처음 10% 스텝 동안은 학습률을 0에서 천천히 올림
    weight_decay=0.0,
    bf16=config["bf16"],
    fp16=config["fp16"],
    gradient_checkpointing=config["gradient_checkpointing"],
    logging_steps=1,
    eval_strategy="epoch",           # epoch마다 평가 데이터 loss 계산
    save_strategy="no",              # 중간 체크포인트는 저장하지 않음 (마지막에 직접 저장)
    report_to="none",                # wandb 등 외부 로깅 끔
    remove_unused_columns=False,     # 우리가 만든 컬럼을 그대로 쓰도록
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=collate,
)

학습 전 평가 loss를 먼저 재 둡니다. 학습 후와 비교할 기준값입니다.

In [ ]:
eval_before = trainer.evaluate()["eval_loss"]
print(f"학습 전 평가 loss: {eval_before:.3f}")

### 학습 시작!

RTX 3060에서는 1~3분, CPU에서는 10~30분 정도 걸립니다. `loss`가 줄어드는 것을 지켜보세요.

In [ ]:
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f"\n학습 시간: {elapsed / 60:.1f}분")
print(f"train loss 평균: {train_result.training_loss:.3f}")
if device == "cuda":
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"학습 중 최대 VRAM 사용량: {peak:.2f} GB")

### loss 곡선 그리기

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 폰트가 있으면 사용 (없으면 그래프의 한글만 □로 보이고 결과에는 영향 없음)
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f.name == font for f in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = font
        plt.rcParams["axes.unicode_minus"] = False
        break

history = trainer.state.log_history
train_logs = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_logs = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_logs), label="train loss", marker=".")
if eval_logs:
    plt.plot(*zip(*eval_logs), label="eval loss", marker="o")
plt.xlabel("업데이트 스텝 (step)")
plt.ylabel("loss")
plt.title("Full 파인튜닝 loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

> 📝 그래프의 한글이 □로 깨지면 한글 폰트가 없는 것입니다. 위 셀은 Windows(맑은 고딕), macOS(AppleGothic) 폰트를 자동으로 찾습니다.
> Linux라면 `sudo apt install fonts-nanum`으로 나눔 폰트를 설치하고 커널을 재시작하세요. 축 이름만 깨질 뿐 결과에는 영향이 없습니다.

**train loss가 계속 줄어드는데 eval loss가 다시 오르기 시작하면** 학습 데이터를 외우기 시작했다는 뜻(**과적합, overfitting**)입니다.
데이터가 42개뿐이라 epoch을 너무 늘리면 금방 과적합됩니다.

## 6. 학습 전후 비교

같은 질문을 다시 물어봅시다.

In [ ]:
eval_after = trainer.evaluate()["eval_loss"]
print(f"평가 loss: 학습 전 {eval_before:.3f} → 학습 후 {eval_after:.3f}\n")

for q in test_questions:
    print(f"Q: {q}")
    print(f"[학습 전] {before_answers[q]}")
    print(f"[학습 후] {ask(model, q)}")
    print("-" * 60)

학습 후에는 처음 보는 질문에도 **냥이 말투로 짧게 답하고 `🐾` 뒤에 멈추는** 것을 볼 수 있습니다.
42개 예시로 지식을 새로 가르친 게 아니라, 이미 알고 있는 지식을 **원하는 형식으로 꺼내는 법**을 가르친 것입니다. 이것이 파인튜닝(SFT)의 핵심입니다.

> ⚠️ 답변 내용이 사실과 다를 수 있습니다. 0.5B는 아주 작은 모델이라 지식이 부족하고, 말투를 배운다고 지식이 늘지는 않습니다.

### catastrophic forgetting 확인

1단계에서 파인튜닝하면 원래 배운 일반 문장 능력이 떨어지는 것(catastrophic forgetting)을 봤습니다.
이번에는 학습률을 작게, 스텝을 적게 잡았기 때문에 일반 문장의 loss가 크게 나빠지지 않아야 합니다.
학습 전 모델과 비교하기 위해 베이스 모델을 다시 불러옵니다.

In [ ]:
general_texts = [
    "대한민국은 동아시아의 한반도 남부에 위치한 나라이다. 수도는 서울이다.",
    "광합성은 식물이 빛 에너지를 이용해 이산화탄소와 물로 포도당을 만드는 과정이다.",
    "The quick brown fox jumps over the lazy dog.",
]

@torch.no_grad()
def text_loss(model, texts):
    model.eval()
    losses = []
    for t in texts:
        batch = tokenizer(t, return_tensors="pt").to(model.device)
        losses.append(model(**batch, labels=batch["input_ids"]).loss.item())
    return sum(losses) / len(losses)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
print(f"일반 문장 loss: 베이스 {text_loss(base_model, general_texts):.3f} → 파인튜닝 후 {text_loss(model, general_texts):.3f}")

del base_model
if device == "cuda":
    torch.cuda.empty_cache()

조금 오르는 것은 정상입니다. 많이 올랐다면 학습률을 낮추거나 epoch을 줄여 보세요.

## 7. 모델 저장하고 다시 불러오기

`save_pretrained`로 저장하면 `from_pretrained`로 그대로 다시 불러올 수 있습니다. Hugging Face Hub에 있는 모델과 똑같은 형식입니다.
Full 파인튜닝은 **모델 전체(약 2GB, fp32)** 를 저장해야 합니다. 3단계 LoRA에서는 이 크기가 수 MB로 줄어드는 것을 보게 됩니다.

> 📁 `outputs/` 폴더는 `.gitignore`에 들어 있어서 git에 올라가지 않습니다.

In [ ]:
SAVE_DIR = OUTPUT_DIR / "final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

size_gb = sum(f.stat().st_size for f in SAVE_DIR.glob("*") if f.is_file()) / 1024**3
print(f"저장 위치: {SAVE_DIR.resolve()}")
print(f"저장 크기: {size_gb:.2f} GB")
for f in sorted(SAVE_DIR.iterdir()):
    print("  ", f.name)

In [ ]:
# 메모리를 비우고 저장된 모델을 새로 불러와서 확인
del trainer, model
if device == "cuda":
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(SAVE_DIR).to(device)
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
print(ask(model, "강아지는 왜 꼬리를 흔들어?"))

## 8. 메모리 절약 팁 정리 (RTX 3060 12GB 기준)

1단계에서 계산한 Full 파인튜닝 메모리(Qwen2.5-0.5B, fp32 + AdamW)는 약 **7.4GB + activation** 이었습니다. 위에서 측정한 최대 VRAM 사용량과 비교해 보세요.
VRAM이 부족해서 `CUDA out of memory`가 나면 위에서부터 차례로 시도합니다.

| 방법 | 설정 | 효과 | 대가 |
|---|---|---|---|
| 배치 줄이고 accumulation 늘리기 | `batch_size=1~2`, `grad_accum_steps` ↑ | activation 메모리 ↓ | 조금 느려짐 |
| 시퀀스 길이 줄이기 | `max_length` ↓ | activation 메모리 ↓ | 긴 예시가 잘림 |
| 혼합 정밀도 | `bf16=True` (또는 `fp16=True`) | activation 메모리 ↓, 속도 ↑ | 거의 없음 |
| Gradient checkpointing | `gradient_checkpointing=True` | activation을 저장하지 않고 역전파 때 다시 계산 → 메모리 크게 ↓ | 약 20~30% 느려짐 |
| 가벼운 옵티마이저 | `optim="adafactor"` 또는 8bit AdamW(`bitsandbytes`) | 옵티마이저 상태(파라미터당 8바이트) ↓ | 학습 특성이 약간 다름 |
| LoRA | 3단계 | gradient와 옵티마이저 상태가 거의 없어짐 | 표현력이 약간 제한됨 |

> 💡 **activation 메모리란?** 순전파 중간 결과를 역전파 때 쓰려고 저장해 두는 것입니다. `배치 크기 × 시퀀스 길이 × 레이어 수`에 비례합니다.
> 특히 Qwen은 어휘가 15만 개라서 마지막 logits(`배치 × 길이 × 151,936`)만 해도 꽤 큽니다.

## 9. 정리

| 개념 | 한 줄 요약 |
|---|---|
| `AutoTokenizer` / `AutoModelForCausalLM` | 이름 하나로 토크나이저와 모델을 불러옴. 원리는 1단계와 같음 |
| 프롬프트 형식 | 학습과 사용 시 같은 형식을 써야 함. 대화 모델은 chat template 사용 |
| EOS 토큰 | 답변 끝에 붙여서 학습해야 멈출 줄 앎 |
| labels `-100` | 프롬프트와 패딩 위치는 loss에서 제외 |
| `Trainer` | 1단계의 학습 루프 + 혼합 정밀도 + accumulation + 평가 + 스케줄러 |
| 혼합 정밀도 | 가중치는 fp32, 계산은 bf16/fp16 |
| Gradient accumulation | 작은 배치를 여러 번 모아 큰 배치 효과 |
| Full 파인튜닝 | 모든 가중치를 학습. 메모리 ≈ 파라미터 × 16바이트 + activation, 저장도 모델 전체 |

### 연습 문제

1. `config`의 `epochs`를 1, 10으로 바꿔 보세요. 말투를 얼마나 빨리 배우나요? eval loss와 일반 문장 loss는 어떻게 변하나요?
2. `lr`을 `1e-4`, `5e-6`으로 바꿔 보세요. 너무 크면, 너무 작으면 어떤 일이 생기나요?
3. `tokenize_example`에서 프롬프트 부분도 loss에 포함하도록(`labels = input_ids` 그대로) 바꿔서 학습해 보세요. 결과가 달라지나요?
4. `data/cat_assistant_train.jsonl`에 나만의 말투나 규칙(예: 항상 세 줄로 답하기)을 가진 예시를 20개쯤 만들어 학습해 보세요.
5. `gradient_checkpointing=True`로 바꾸고 최대 VRAM 사용량과 학습 시간을 비교해 보세요.

### 다음 단계

**03. LoRA 파인튜닝**에서는 같은 데이터, 같은 과제를 LoRA로 학습합니다.
학습하는 파라미터 수, VRAM 사용량, 저장 크기, 결과를 이번 Full 파인튜닝과 나란히 비교해 보겠습니다.